## 1. Kütüphane İmportları ve Konfigürasyon

Bu bölüm, training notebook'unda kullanılan tüm sınıfları ve kaydedilen model bileşenlerini yükler.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import pickle
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Kütüphaneler başarıyla yüklendi!")
print(f"Çalışma dizini: {os.getcwd()}")
print(f"Python versiyonu: {sys.version.split()[0]}")

Kütüphaneler başarıyla yüklendi!
Çalışma dizini: c:\Users\alper\OneDrive\Masaüstü\dersler\yap470\topic_calssifier\2.veriseti
Python versiyonu: 3.11.9


## 3. Bileşenleri Yükle ve CSV Test Verileri Hazırla

In [2]:
# ============================================================================
# YÖNTEM 2 (RNN) MODELİNİ YÜKLE VE TEST ET
# ============================================================================

print("Sistem kontrolleri...")
print(f"Çalışma dizini: {os.getcwd()}")
print(f"Python versiyonu: {sys.version.split()[0]}")

# Gerekli dosyalar: models2 klasörü
models2_dir = "models2"
if not os.path.exists(models2_dir):
    print(f"models2 klasörü bulunamadı!")
    print("Çözüm: yontem_2.ipynb notebook'unu çalıştırın!")
    sys.exit()

# Tokenizer ve Config Yükle
with open(os.path.join(models2_dir, "tokenizer_rnn.pkl"), "rb") as f:
    tokenizer = pickle.load(f)
with open(os.path.join(models2_dir, "rnn_model_config.pkl"), "rb") as f:
    model_config = pickle.load(f)

# Modeli Yükle
rnn_model = tf.keras.models.load_model(os.path.join(models2_dir, "rnn_model_final.h5"))

# Test Verisini Hazırla
test_file_path = "archive/test.csv"
test_df = pd.read_csv(test_file_path, header=None)
test_df.columns = ["label", "title", "description"]
test_df = test_df[test_df["label"] != "Class Index"]
test_df = test_df[test_df["label"] != "label"]
test_df = test_df[pd.to_numeric(test_df["label"], errors='coerce').notna()]
test_df["label"] = test_df["label"].astype(int)
test_df["combined_text"] = test_df["title"].fillna("") + " " + test_df["description"].fillna("")

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

test_df["cleaned_text"] = test_df["combined_text"].apply(clean_text)
test_df = test_df[test_df["cleaned_text"].str.len() > 0]

# Tokenize ve Pad
sequences = tokenizer.texts_to_sequences(test_df["cleaned_text"])
X_test = pad_sequences(sequences, maxlen=model_config["max_sequence_length"], padding='post', truncating='post')
y_test = test_df["label"].values

# Tahmin ve Değerlendirme
print("Model test verisi üzerinde tahmin yapıyor...")
y_pred_proba = rnn_model.predict(X_test, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Analiz ve manuel test için gerekli değişkenler
test_texts = test_df["cleaned_text"].tolist()
test_labels = y_test.tolist()

Sistem kontrolleri...
Çalışma dizini: c:\Users\alper\OneDrive\Masaüstü\dersler\yap470\topic_calssifier\2.veriseti
Python versiyonu: 3.11.9


Model test verisi üzerinde tahmin yapıyor...
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 48s 22ms/step
Accuracy: 0.9869285714285714

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97      5000
           1       0.99      0.98      0.99      5000
           2       0.97      0.98      0.98      5000
           3       0.99      0.99      0.99      5000
           4       0.99      0.98      0.98      5000
           5       0.99      0.99      0.99      5000
           6       0.98      0.98      0.98      5000
           7       0.99      0.99      0.99      5000
           8       1.00      1.00      1.00      5000
           9       1.00      0.99      0.99      5000
          10       0.99      0.99      0.99      5000
          11       0.99      0.99      0.99      5000
          12       1.00      0.98      0.99      5000
          13       0.98      0.99      0.98      5000

    accuracy                           0.99 

## 4. Manuel Test - Kendi Metninizi Test Edin

Bu bölümde kendi yazdığınız metinleri test edebilirsiniz. Aşağıdaki hücreyi düzenleyerek istediğiniz metni ve beklenen kategoriyi belirleyebilirsiniz.

**Kategoriler:**
- 0: Company (Şirket/Organizasyon)
- 1: Sports (Spor)
- 2: Business (İş/Ekonomi)
- 3: Science/Tech (Bilim/Teknoloji)
- 4: Politics (Politika)
- 5: Education (Eğitim)
- 6: Health (Sağlık)
- 7: Geography (Coğrafya)
- 8: Art/Media (Sanat/Medya)
- 9: Nature/Biology (Doğa/Biyoloji)
- 10: Biology (Biyoloji detay)
- 11: Music (Müzik)
- 12: Film (Film)
- 13: Literature (Edebiyat)

In [5]:
# MANUEL TEST - KENDİ METNİNİZİ BURAYA YAZIN
print("="*60)
print("MANUEL METİN TESTİ")
print("="*60)

# BURASI SİZİN DÜZENLEYECEĞİNİZ BÖLÜM
my_text = "Apple reported record quarterly earnings beating analyst expectations"
my_expected_category = 2  # 0: Company, 1: Sports, 2: Business, 3: Science/Tech, vs.

category_names = [
    "Company", "Sports", "Business", "Science/Tech", "Politics", "Education", "Health",
    "Geography", "Art/Media", "Nature/Biology", "Biology", "Music", "Film", "Literature"
 ]

def predict_text(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=model_config["max_sequence_length"], padding='post', truncating='post')
    proba = rnn_model.predict(padded)
    pred = np.argmax(proba, axis=1)[0]
    return pred, proba[0]

if my_text and my_text.strip():
    pred_idx, proba = predict_text(my_text)
    print(f"Test Metni: {my_text}")
    print(f"Beklenen Kategori: {category_names[my_expected_category]}")
    print(f"Tahmin Edilen Kategori: {category_names[pred_idx]}")
    print(f"Tahmin Doğru mu?: {'EVET' if pred_idx == my_expected_category else 'HAYIR'}")
    print(f"Kategori Olasılıkları: {np.round(proba, 3)}")
else:
    print("Lütfen 'my_text' değişkenini yukarıda düzenleyin!")

# Örnek testler (isteğe bağlı)
sample_tests = [
    ("Manchester United won the Champions League final against Real Madrid", 1),
    ("NASA successfully launched a new rover to Mars surface", 3), 
    ("European Union announces new trade sanctions against Russia", 4),
    ("Tesla stock price surged after strong delivery numbers", 2),
    ("Scientists discovered a new species of dinosaur in Argentina", 9),
    ("Harvard University announces new scholarship program", 5),
    ("WHO reports new breakthrough in cancer treatment", 6),
    ("Amazon rainforest deforestation reaches record levels", 9),
    ("Netflix releases new documentary about climate change", 12),
    ("Beethoven's lost symphony discovered in German archive", 11)
 ]

print("\nÖRNEK TESTLER")
for i, (sample_text, expected_cat) in enumerate(sample_tests[:10], 1):
    pred_idx, _ = predict_text(sample_text)
    print(f"{i}. [{category_names[expected_cat]}] {sample_text[:60]}...")
    print(f"   Tahmin: {category_names[pred_idx]} | Doğru mu?: {'EVET' if pred_idx == expected_cat else 'HAYIR'}")

MANUEL METİN TESTİ
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Test Metni: Apple reported record quarterly earnings beating analyst expectations
Beklenen Kategori: Business
Tahmin Edilen Kategori: Literature
Tahmin Doğru mu?: HAYIR
Kategori Olasılıkları: [0.068 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.931]

ÖRNEK TESTLER
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1. [Sports] Manchester United won the Champions League final against Rea...
   Tahmin: Company | Doğru mu?: HAYIR
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
2. [Science/Tech] NASA successfully launched a new rover to Mars surface...
   Tahmin: Education | Doğru mu?: HAYIR
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
3. [Politics] European Union announces new trade sanctions against Russia...
   Tahmin: Company | Doğru mu?: HAYIR
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
4. [Business] Tesla stock price surged after strong delivery numbers...
   Tahmin: Company | Doğru mu?: HAYIR
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
5. [Nat

## 5. Detaylı Analiz - Comprehensive Model Evaluation

Bu bölümde tüm modellerin performansını detaylı şekilde analiz ediyoruz:

### Analizler:
- **Confusion Matrix**: Her model için karışıklık matrisi
- **Kategori Bazında Doğruluk**: Hangi kategoride hangi model en başarılı
- **Model Hızları**: Prediction süreleri karşılaştırması  
- **En İyi Model Seçimi**: Farklı kriterler için öneriler
- **Detaylı Performans Metrikleri**: Precision, Recall, F1-Score

In [4]:
# DETAYLI ANALİZ - SADECE RNN MODELİ İÇİN

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

if not test_texts or not test_labels:
    print("Test verileri bulunamadı! Önce test hücresini çalıştırın.")
else:
    print("="*60)
    print("DETAYLI ANALİZ - RNN MODELİ")
    print("="*60)
    
    # Toplu tahmin
    sequences = tokenizer.texts_to_sequences(test_texts)
    X = pad_sequences(sequences, maxlen=model_config["max_sequence_length"], padding='post', truncating='post')
    y_true = np.array(test_labels)
    y_pred_proba = rnn_model.predict(X, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Genel performans
    acc = accuracy_score(y_true, y_pred)
    print(f"Genel Accuracy: {acc:.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    category_names = [
        "Company", "Sports", "Business", "Science/Tech", "Politics", "Education", "Health",
        "Geography", "Art/Media", "Nature/Biology", "Biology", "Music", "Film", "Literature"
    ]
    present_labels = sorted(list(set(y_true)))
    present_names = [category_names[i] for i in present_labels]
    print("\nConfusion Matrix:")
    cm_df = pd.DataFrame(cm, index=present_names, columns=present_names)
    print(cm_df)

    # Kategori bazında doğruluk
    print("\nKategori Bazında Doğruluk:")
    for idx, name in zip(present_labels, present_names):
        mask = y_true == idx
        cat_acc = accuracy_score(y_true[mask], y_pred[mask])
        print(f"{name}: {cat_acc:.3f} ({mask.sum()} sample)")

DETAYLI ANALİZ - RNN MODELİ
Genel Accuracy: 0.9869

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97      5000
           1       0.99      0.98      0.99      5000
           2       0.97      0.98      0.98      5000
           3       0.99      0.99      0.99      5000
           4       0.99      0.98      0.98      5000
           5       0.99      0.99      0.99      5000
           6       0.98      0.98      0.98      5000
           7       0.99      0.99      0.99      5000
           8       1.00      1.00      1.00      5000
           9       1.00      0.99      0.99      5000
          10       0.99      0.99      0.99      5000
          11       0.99      0.99      0.99      5000
          12       1.00      0.98      0.99      5000
          13       0.98      0.99      0.98      5000

    accuracy                           0.99     70000
   macro avg       0.99      0.99      0.99     70000
weig